# Modelado Pyspark

In [1]:
from pyspark.sql import SparkSession

# 1. Configuración de rutas (Basado en tu test exitoso)
jar_path = "/home/alejo/spark_jars/rapids-4-spark_2.12-24.02.0.jar"

# 2. Inicializar Sesión de Spark con soporte para GPU
spark = SparkSession.builder \
    .appName("Proyecto_Integrador_PySpark") \
    .config("spark.plugins", "com.nvidia.spark.SQLPlugin") \
    .config("spark.driver.extraClassPath", jar_path) \
    .config("spark.executor.extraClassPath", jar_path) \
    .config("spark.rapids.sql.enabled", "true") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.rapids.sql.rowBasedUDF.enabled", "true") \
    .config("spark.rapids.sql.csv.read.decimal.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "96") \
    .getOrCreate()

# 3. Cargar el dataset train.gz
# header=True: Usa la primera fila como nombres de columnas
# inferSchema=True: Spark intentará detectar si los datos son int, float, etc.
# Nota: Con 40M de filas, inferSchema puede tardar un poco.
path_to_file = "train.csv"

df_spark = spark.read.csv(path_to_file, header=True, inferSchema=True)

# 4. Verificación inicial
print(f"Dimensiones estimadas: {df_spark.count()} filas")
df_spark.printSchema()
df_spark.show(5)

your 131072x1 screen size is bogus. expect trouble
26/03/19 21:26:37 WARN Utils: Your hostname, DESKTOP-65QCQ4G resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/19 21:26:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 21:26:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/19 21:26:38 WARN RapidsPluginUtils: RAPIDS Accelerator 24.02.0 using cudf 24.02.1.
26/03/19 21:26:38 WARN RapidsPluginUtils: RAPIDS Accelerator is enabled, to disable GPU support set `spark.rapids.sql.enabled` to false.
26/03/19 21:26:38 WARN RapidsPluginUtils: spark.rapids.sql.explain is set to `NOT_ON_GPU`. Set it to 'NONE' to suppress the diagnostics logging about the query placement on the GPU.
26/03/19 21:26:46 WARN GpuOve

Dimensiones estimadas: 40428967 filas
root
 |-- id: decimal(20,0) (nullable = true)
 |-- click: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- C1: integer (nullable = true)
 |-- banner_pos: integer (nullable = true)
 |-- site_id: string (nullable = true)
 |-- site_domain: string (nullable = true)
 |-- site_category: string (nullable = true)
 |-- app_id: string (nullable = true)
 |-- app_domain: string (nullable = true)
 |-- app_category: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- device_ip: string (nullable = true)
 |-- device_model: string (nullable = true)
 |-- device_type: integer (nullable = true)
 |-- device_conn_type: integer (nullable = true)
 |-- C14: integer (nullable = true)
 |-- C15: integer (nullable = true)
 |-- C16: integer (nullable = true)
 |-- C17: integer (nullable = true)
 |-- C18: integer (nullable = true)
 |-- C19: integer (nullable = true)
 |-- C20: integer (nullable = true)
 |-- C21: integer (nullable = true)

+---

In [2]:
# ==============================================================================
# CELDA 1 — Imports y definición de columnas base
# NOTA: franja_horaria, hora_del_dia y dia_semana se añaden al final de Celda 2,
# después de ser creadas con withColumn. No se pueden listar aquí porque aún
# no existen como columnas en df_spark.
# ==============================================================================

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import time
import itertools

# Suprimir WARN de GPU y MemoryStore — solo muestra ERRORs reales
spark.sparkContext.setLogLevel("ERROR")

target_col = "click"

# --- Columnas categóricas base (ya existen en df_spark al cargarlo) ---
# Decisiones basadas en cardinalidad y V de Cramér:
#   - site_id (4737), site_domain (7745): drop → redundantes con site_category (26)
#   - app_id (8552), app_domain (559):    drop → redundantes con app_category (36)
#   - device_id (2.6M), device_ip (6.7M): drop → near-unique, ruido puro
#   - device_model (8251): drop → V=0.84 con device_type, alta cardinalidad
#   - C15(8),C16(9),C17(435),C18(4),C21(60): drop → V≥0.99 con C14, redundantes
categorical_cols = [
    "C1",               # 7 categorías → señal independiente
    "banner_pos",       # pocos valores únicos
    "device_type",      # pocos valores únicos
    "device_conn_type", # pocos valores únicos
    "site_category",    # 26 categorías → señal útil
    "app_category",     # 36 categorías → señal útil
    # franja_horaria se añade al final de Celda 2
]

# --- Columnas numéricas base (target encoded, se añaden en Celda 3 tras TE) ---
# C14 (2626), C19 (68), C20 (172): alta cardinalidad → Target Encoding
# hora_del_dia y dia_semana se añaden al final de Celda 2
numerical_cols = [
    "C14_te",   # target encoded (reemplaza C14 y sus redundantes C15,C16,C17,C18,C21)
    "C19_te",   # target encoded (señal independiente de C14)
    "C20_te",   # target encoded (V=0.41 con C14, claramente independiente)
    # hora_del_dia y dia_semana se añaden al final de Celda 2
]

print(f"Columnas categóricas base ({len(categorical_cols)}): {categorical_cols}")
print(f"Columnas numéricas base   ({len(numerical_cols)}): {numerical_cols}")




Columnas categóricas base (6): ['C1', 'banner_pos', 'device_type', 'device_conn_type', 'site_category', 'app_category']
Columnas numéricas base   (3): ['C14_te', 'C19_te', 'C20_te']


In [3]:
# ==============================================================================
# CELDA 2 — Feature engineering temporal + completar listas de columnas
# ==============================================================================

# --- Features temporales desde 'hour' (formato YYMMDDHH, ej: 14102100) ---
# Sentencias separadas: cada columna debe existir antes de usarse en la siguiente
df_spark = df_spark.withColumn("hora_del_dia", (F.col("hour") % 100).cast("int"))

# Reconstruir fecha para extraer día de la semana
# (más informativo que día del mes para CTR — comportamiento weekday vs weekend)
# dayofweek: 1=Domingo, 2=Lunes, ..., 7=Sábado
df_spark = df_spark.withColumn(
    "fecha_str",
    F.concat(
        F.lit("20"),
        F.lpad((F.floor(F.col("hour") / 1000000)).cast("string"),     2, "0"),  # YY
        F.lpad((F.floor(F.col("hour") / 10000) % 100).cast("string"), 2, "0"),  # MM
        F.lpad((F.floor(F.col("hour") / 100)   % 100).cast("string"), 2, "0"),  # DD
    )
)
df_spark = df_spark.withColumn(
    "dia_semana",
    F.dayofweek(F.to_date(F.col("fecha_str"), "yyyyMMdd"))
)
df_spark = df_spark.drop("fecha_str")  # columna auxiliar, no se necesita más

# franja_horaria depende de hora_del_dia — debe crearse después
df_spark = df_spark.withColumn(
    "franja_horaria",
    F.when((F.col("hora_del_dia") >= 0)  & (F.col("hora_del_dia") < 6),  "madrugada")
    .when((F.col("hora_del_dia") >= 6)  & (F.col("hora_del_dia") < 12), "manana")
    .when((F.col("hora_del_dia") >= 12) & (F.col("hora_del_dia") < 18), "tarde")
    .otherwise("noche")
)

# --- Ahora que las columnas temporales existen, se añaden a las listas ---
categorical_cols.append("franja_horaria")   # 4 valores: madrugada/manana/tarde/noche
numerical_cols.extend(["hora_del_dia", "dia_semana"])

print(f"Columnas categóricas completas ({len(categorical_cols)}): {categorical_cols}")
print(f"Columnas numéricas completas   ({len(numerical_cols)}): {numerical_cols}")

# --- Distribución de clases ---
total       = df_spark.count()
count_click = df_spark.filter(F.col("click") == 1).count()
count_no    = total - count_click
global_mean = count_click / total  # tasa global de clics — usada como fillna en target encoding

print(f"\nTotal filas:    {total:,}")
print(f"click=1 (clic): {count_click:,}  ({100*count_click/total:.1f}%)")
print(f"click=0 (no):   {count_no:,}  ({100*count_no/total:.1f}%)")
print(f"Tasa global de clic: {global_mean:.4f}")


Columnas categóricas completas (7): ['C1', 'banner_pos', 'device_type', 'device_conn_type', 'site_category', 'app_category', 'franja_horaria']
Columnas numéricas completas   (5): ['C14_te', 'C19_te', 'C20_te', 'hora_del_dia', 'dia_semana']



Total filas:    40,428,967
click=1 (clic): 6,865,066  (17.0%)
click=0 (no):   33,563,901  (83.0%)
Tasa global de clic: 0.1698


In [4]:
# ==============================================================================
# CELDA 3a — Split, undersampling 
# ==============================================================================

CKPT = "/home/alejo/spark_project/data"  # carpeta permanente, no /tmp
import os
os.makedirs(CKPT, exist_ok=True)

# Escribir df_spark a parquet ANTES del split y undersampling
# Trunca el plan acumulado desde Cell 0: CSV read → inferSchema → withColumns (Cell 2)
# Sin esto, filter+sample+union arrastran todo ese plan y explotan el heap
df_spark.write.mode("overwrite").parquet(f"{CKPT}/df_spark")

del df_spark
spark.catalog.clearCache()
spark.sparkContext._jvm.System.gc()
time.sleep(2)
print("SE LIMPIO EL CACHE")

df_spark = spark.read.parquet(f"{CKPT}/df_spark")

# --- Split train/test ANTES de cualquier transformación (evita data leakage) ---
train_data, test_data = df_spark.randomSplit([0.8, 0.2], seed=42)

# Contar clases en un solo job
train_counts      = train_data.groupBy("click").count().collect()
count_map         = {int(r["click"]): r["count"] for r in train_counts}
count_no_train    = count_map.get(0, 0)
count_click_train = count_map.get(1, 0)
total_train       = count_no_train + count_click_train

print(f"Train completo: {total_train:,} filas  "
      f"(click=0: {count_no_train:,} | click=1: {count_click_train:,})")
print(f"Test  completo: aprox. {int(total * 0.2):,} filas")

# --- Undersampling SOLO en train ---
# Test se mantiene sin tocar → refleja distribución real (83/17)
# Threshold tuning en Celda 4 recalibra al mundo real tras entrenar en 50/50
MAX_POR_CLASE = 5_000_000

frac_0 = min(1.0, MAX_POR_CLASE / count_no_train)
frac_1 = min(1.0, MAX_POR_CLASE / count_click_train)

df_train_0     = train_data.filter(F.col("click") == 0).sample(fraction=frac_0, seed=42)
df_train_1     = train_data.filter(F.col("click") == 1).sample(fraction=frac_1, seed=42)
train_balanced = df_train_0.union(df_train_1)

"""
print(f"\nTrain balanceado: {train_balanced.count():,} filas")
print("Distribución tras undersampling:")
train_balanced.groupBy("click").count().orderBy("click").show()
"""

# Escribir a parquet trunca el plan acumulado desde el CSV load
# (withColumns → split → filter → sample → union)
# Sin esto, Cell 3b hereda todo ese plan encima del groupBy de target encoding
train_balanced.write.mode("overwrite").parquet(f"{CKPT}/train_balanced")
test_data.write.mode("overwrite").parquet(f"{CKPT}/test_data")

del df_spark, train_data, df_train_0, df_train_1, train_balanced, test_data
spark.catalog.clearCache()
spark.sparkContext._jvm.System.gc()
time.sleep(2)

train_balanced = spark.read.parquet(f"{CKPT}/train_balanced")
test_data      = spark.read.parquet(f"{CKPT}/test_data")

print(f"Train balanceado: aprox. {MAX_POR_CLASE * 2:,} filas")
print("Parquet escrito. Plan truncado.")

SE LIMPIO EL CACHE


Train completo: 32,341,280 filas  (click=0: 26,848,695 | click=1: 5,492,585)
Test  completo: aprox. 8,085,793 filas


Train balanceado: aprox. 10,000,000 filas
Parquet escrito. Plan truncado.


In [5]:
# ==============================================================================
# CELDA 3b — Target Encoding sin joins, sin toPandas() sobre datos grandes
# ==============================================================================

# Calcular la tasa media de clics por categoría con groupBy (solo en Spark)
# collect() trae al driver SOLO el resultado agregado — 2626 filas para C14,
# 68 para C19, 172 para C20. Nada de 10M filas al heap.
for col_orig, col_te in [("C14", "C14_te"), ("C19", "C19_te"), ("C20", "C20_te")]:
    te_rows = (train_balanced
               .groupBy(col_orig)
               .agg(F.mean("click").alias(col_te))
               .collect())

    # Convertir el resultado pequeño a dict en el driver
    mapping = {str(r[col_orig]): float(r[col_te]) for r in te_rows}
    print(f"  {col_orig}: {len(mapping)} categorías codificadas")

    # Aplicar como UDF — sin join, sin shuffle, sin mover datos grandes al driver
    te_udf = F.udf(lambda x: mapping.get(str(x), global_mean), DoubleType())
    train_balanced = train_balanced.withColumn(col_te, te_udf(F.col(col_orig)))
    test_data      = test_data.withColumn(col_te, te_udf(F.col(col_orig)))

print("Target encoding completado.")

# Truncar plan de nuevo antes del pipeline fit
train_balanced.write.mode("overwrite").parquet(f"{CKPT}/train_balanced_te")
test_data.write.mode("overwrite").parquet(f"{CKPT}/test_data_te")

del train_balanced, test_data
spark.catalog.clearCache()

train_balanced = spark.read.parquet(f"{CKPT}/train_balanced_te")
test_data      = spark.read.parquet(f"{CKPT}/test_data_te")

print("Target encoding completado. Plan truncado.")

  C14: 2525 categorías codificadas
  C19: 68 categorías codificadas
  C20: 167 categorías codificadas
Target encoding completado.


Target encoding completado. Plan truncado.


In [6]:
# ==============================================================================
# CELDA 3c — Pipeline fit y transform
# ==============================================================================

# --- StringIndexer para cada columna categórica ---
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in categorical_cols
]

# --- OneHotEncoder ---
encoders = [
    OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_ohe")
    for c in categorical_cols
]

# --- VectorAssembler: OHE + numéricas → 'features_raw' ---
assembler_inputs = [c + "_ohe" for c in categorical_cols] + numerical_cols
assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features_raw",
    handleInvalid="keep"
)

# --- StandardScaler ---
# withMean=False: vectores sparse de OHE no admiten centrado
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

# --- Pipeline de preprocesamiento ---
preprocessing_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])

print("Ajustando pipeline sobre train_balanced...")
preprocessor_model = preprocessing_pipeline.fit(train_balanced)

train_assembled = preprocessor_model.transform(train_balanced).select("features", target_col)
test_assembled  = preprocessor_model.transform(test_data).select("features", target_col)

train_assembled.write.mode("overwrite").parquet(f"{CKPT}/train_assembled")
test_assembled.write.mode("overwrite").parquet(f"{CKPT}/test_assembled")

del train_balanced, test_data, preprocessor_model
spark.catalog.clearCache()

train_assembled = spark.read.parquet(f"{CKPT}/train_assembled")
test_assembled  = spark.read.parquet(f"{CKPT}/test_assembled")

input_size = int(train_assembled.select("features").head()["features"].size)
print(f"Dimensión del vector de features: {input_size}")
print("Preprocesamiento completado.")




Ajustando pipeline sobre train_balanced...


Dimensión del vector de features: 87
Preprocesamiento completado.


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Proyecto_Integrador_PySpark") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.sql.shuffle.partitions", "24") \
    .getOrCreate()

your 131072x1 screen size is bogus. expect trouble
26/03/20 11:51:48 WARN Utils: Your hostname, DESKTOP-65QCQ4G resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/20 11:51:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/20 11:51:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import time
import itertools

spark.sparkContext.setLogLevel("ERROR")
target_col = "click"
CKPT = "/home/alejo/spark_project/data"
train_assembled = spark.read.parquet(f"{CKPT}/train_assembled").repartition(24)
test_assembled  = spark.read.parquet(f"{CKPT}/test_assembled").repartition(24)
input_size = int(train_assembled.select("features").head()["features"].size)
print(f"input_size={input_size}")

input_size=87


In [3]:
# ==============================================================================
# CELDA 4 — SECCIÓN 11.12.8: Modelado MLP (fit directo, sin CrossValidator)
# CrossValidator se elimina: entrena el modelo 3x por configuración, triplicando
# la presión de memoria en una sesión local de Spark.
# Con 10M filas balanceadas, un único fit es estadísticamente robusto.
# ==============================================================================

# --- Grid de hiperparámetros según el PDF ---
# Solver 'gd': más estable que l-bfgs en datasets grandes
# stepSize 0.1 incluido: con 10M filas y gd puede converger
# Prefijo 'grid_' para evitar colisión con variables escalares de celdas anteriores
grid_layers = [
    [input_size, 10,  2],
    [input_size, 50,  2],
    [input_size, 100, 2],
]
grid_step_sizes = [0.1, 0.01]
grid_max_iters  = [100, 200]

# Verificación defensiva
assert isinstance(grid_layers, list),     f"grid_layers debe ser lista, es {type(grid_layers)}"
assert isinstance(grid_step_sizes, list), f"grid_step_sizes debe ser lista, es {type(grid_step_sizes)}"
assert isinstance(grid_max_iters, list),  f"grid_max_iters debe ser lista, es {type(grid_max_iters)}"
assert isinstance(input_size, int),       f"input_size debe ser int, es {type(input_size)}"
n_configs = len(grid_layers) * len(grid_step_sizes) * len(grid_max_iters)
print(f"Grid OK — input_size={input_size}, "
      f"layers={[l[1] for l in grid_layers]}, "
      f"steps={grid_step_sizes}, iters={grid_max_iters}")
print(f"Total configuraciones: {n_configs} entrenamientos")

# --- Evaluador AUC-ROC ---
evaluador_auc = BinaryClassificationEvaluator(
    labelCol=target_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)


# --- Función para encontrar el mejor threshold ---
def encontrar_mejor_threshold(predictions, pasos=15):  # 30 → 15
    sample = (predictions
              .select("probability", target_col)
              .sample(fraction=0.05, seed=42)  # 10% → 5%, suficiente con 8M filas
              .collect())

    probs = [r["probability"][1] for r in sample]
    if not probs:
        return 0.5

    max_prob = max(probs)
    if max_prob < 0.1:
        print(f"  AVISO: modelo degenerado (max_prob={max_prob:.4f}). Threshold=0.50.")
        return 0.5

    probs_sorted = sorted(probs)
    # Percentil 20-80 en vez de 5-95 — más enfocado donde están las predicciones reales
    low  = probs_sorted[int(len(probs_sorted) * 0.20)]
    high = probs_sorted[int(len(probs_sorted) * 0.80)]
    step = (high - low) / pasos if high > low else 0.01

    mejor_f1        = 0.0
    mejor_threshold = 0.5

    for t in [low + i * step for i in range(pasos + 1)]:
        tp = sum(1 for r in sample if r["probability"][1] >= t and r[target_col] == 1)
        fp = sum(1 for r in sample if r["probability"][1] >= t and r[target_col] == 0)
        fn = sum(1 for r in sample if r["probability"][1] <  t and r[target_col] == 1)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)

        if f1 > mejor_f1:
            mejor_f1        = f1
            mejor_threshold = t

    print(f"  Mejor threshold: {mejor_threshold:.4f}  (F1 muestra: {mejor_f1:.4f})")
    return mejor_threshold


# --- Función de evaluación ---
def evaluar_modelo(predictions, nombre_config, threshold=0.5):
    print(f"\n{'='*58}")
    print(f"  Resultados: {nombre_config}")
    print(f"{'='*58}")

    adjust_pred = F.udf(
        lambda prob: 1.0 if prob[1] >= threshold else 0.0,
        DoubleType()
    )
    predictions = predictions.withColumn("prediction", adjust_pred(F.col("probability")))

    cm_rows = (predictions
               .groupBy(target_col, "prediction")
               .count()
               .collect())
    cm_map = {
        (int(r[target_col]), int(r["prediction"])): r["count"]
        for r in cm_rows
    }
    tn = cm_map.get((0, 0), 0)
    fp = cm_map.get((0, 1), 0)
    fn = cm_map.get((1, 0), 0)
    tp = cm_map.get((1, 1), 0)

    total_pred = tn + fp + fn + tp
    accuracy  = (tp + tn) / total_pred if total_pred > 0 else 0.0
    precision = tp / (tp + fp)         if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn)         if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    auc_roc   = evaluador_auc.evaluate(predictions)

    print(f"  Threshold:  {threshold:.4f}")
    print(f"  Accuracy:   {accuracy:.4f}")
    print(f"  Precision:  {precision:.4f}")
    print(f"  Recall:     {recall:.4f}")
    print(f"  F1-Score:   {f1:.4f}")
    print(f"  AUC-ROC:    {auc_roc:.4f}")
    print(f"\n  Matriz de Confusión (filas=real, cols=predicho):")
    print(f"              Pred 0    Pred 1")
    print(f"  Real 0:  {tn:>8,}  {fp:>8,}")
    print(f"  Real 1:  {fn:>8,}  {tp:>8,}")

    return {
        "config":    nombre_config,
        "threshold": threshold,
        "accuracy":  accuracy,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc_roc":   auc_roc,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


# --- Loop de entrenamiento (fit directo, sin CrossValidator) ---
resultados_totales = []

for layers, step, max_it in itertools.product(grid_layers, grid_step_sizes, grid_max_iters):
    nombre = f"layers={layers[1]} | stepSize={step} | maxIter={max_it}"
    print(f"\n>>> Entrenando: {nombre}")

    mlp = MultilayerPerceptronClassifier(
        featuresCol="features",
        labelCol=target_col,
        layers=layers,
        solver="gd",
        stepSize=step,
        maxIter=max_it,
        blockSize=512,
        seed=42
    )

    # Medir tiempo de entrenamiento
    t_inicio_train = time.time()
    mlp_model      = mlp.fit(train_assembled)
    tiempo_train   = time.time() - t_inicio_train

    # Medir tiempo de predicción
    t_inicio_pred = time.time()
    predictions   = mlp_model.transform(test_assembled)
    tiempo_pred = time.time() - t_inicio_pred

    print(f"  Tiempo entrenamiento: {tiempo_train:.2f} s")
    print(f"  Tiempo predicción:    {tiempo_pred:.2f} s")

    threshold = encontrar_mejor_threshold(predictions)
    resultado = evaluar_modelo(predictions, nombre, threshold=threshold)
    resultado["tiempo_train"] = tiempo_train
    resultado["tiempo_pred"]  = tiempo_pred
    resultados_totales.append(resultado)

Grid OK — input_size=87, layers=[10, 50, 100], steps=[0.1, 0.01], iters=[100, 200]
Total configuraciones: 12 entrenamientos

>>> Entrenando: layers=10 | stepSize=0.1 | maxIter=100


  Tiempo entrenamiento: 223.39 s
  Tiempo predicción:    0.11 s


  Mejor threshold: 0.5120  (F1 muestra: 0.3577)

  Resultados: layers=10 | stepSize=0.1 | maxIter=100


  Threshold:  0.5120
  Accuracy:   0.5848
  Precision:  0.2427
  Recall:     0.6823
  F1-Score:   0.3580
  AUC-ROC:    0.6402

  Matriz de Confusión (filas=real, cols=predicho):
              Pred 0    Pred 1
  Real 0:  3,793,414  2,921,792
  Real 1:   436,064   936,417

>>> Entrenando: layers=10 | stepSize=0.1 | maxIter=200


  Tiempo entrenamiento: 397.96 s
  Tiempo predicción:    0.15 s


...

  Mejor threshold: 0.4644  (F1 muestra: 0.3021)

  Resultados: layers=100 | stepSize=0.01 | maxIter=100


  Threshold:  0.4644
  Accuracy:   0.3243
  Precision:  0.1837
  Recall:     0.8656
  F1-Score:   0.3030
  AUC-ROC:    0.5551

  Matriz de Confusión (filas=real, cols=predicho):
              Pred 0    Pred 1
  Real 0:  1,435,001  5,280,205
  Real 1:   184,527  1,187,954

>>> Entrenando: layers=100 | stepSize=0.01 | maxIter=200


  Tiempo entrenamiento: 2387.45 s
  Tiempo predicción:    0.14 s


  Mejor threshold: 0.4926  (F1 muestra: 0.3177)

  Resultados: layers=100 | stepSize=0.01 | maxIter=200


  Threshold:  0.4926
  Accuracy:   0.4279
  Precision:  0.1992
  Recall:     0.7854
  F1-Score:   0.3178
  AUC-ROC:    0.5798

  Matriz de Confusión (filas=real, cols=predicho):
              Pred 0    Pred 1
  Real 0:  2,382,582  4,332,624
  Real 1:   294,562  1,077,919


In [4]:
# ==============================================================================
# CELDA 5 — Tabla comparativa de resultados — SECCIÓN 11.12.10
# ==============================================================================

print(f"{'Configuración':<42} {'Thresh':>7} {'F1':>6} {'AUC':>6} "
      f"{'Prec':>6} {'Rec':>6} {'T.Train':>9} {'T.Pred':>8}")
print("-"*100)
for r in resultados_totales:
    print(f"{r['config']:<42} {r['threshold']:>7.4f} {r['f1']:>6.4f} {r['auc_roc']:>6.4f} "
          f"{r['precision']:>6.4f} {r['recall']:>6.4f} "
          f"{r['tiempo_train']:>8.1f}s {r['tiempo_pred']:>7.1f}s")

mejor = max(resultados_totales, key=lambda x: x["f1"])
print(f"\n>>> Mejor configuración (por F1 en test): {mejor['config']}")
print(f"    Threshold={mejor['threshold']:.4f} | F1={mejor['f1']:.4f} | AUC={mejor['auc_roc']:.4f}")

Configuración                               Thresh     F1    AUC   Prec    Rec   T.Train   T.Pred
----------------------------------------------------------------------------------------------------
layers=10 | stepSize=0.1 | maxIter=100      0.5120 0.3580 0.6402 0.2427 0.6823    223.4s     0.1s
layers=10 | stepSize=0.1 | maxIter=200      0.5257 0.3678 0.6522 0.2557 0.6549    398.0s     0.2s
layers=10 | stepSize=0.01 | maxIter=100     0.4966 0.3031 0.5992 0.2008 0.6181    214.2s     0.1s
layers=10 | stepSize=0.01 | maxIter=200     0.4888 0.3172 0.6112 0.2045 0.7066    395.7s     0.2s
layers=50 | stepSize=0.1 | maxIter=100      0.5431 0.3667 0.6531 0.2667 0.5867    693.1s     0.1s
layers=50 | stepSize=0.1 | maxIter=200      0.5446 0.3727 0.6730 0.2661 0.6216   1339.9s     0.1s
layers=50 | stepSize=0.01 | maxIter=100     0.4550 0.3058 0.5339 0.1853 0.8738    697.7s     0.1s
layers=50 | stepSize=0.01 | maxIter=200     0.4823 0.3220 0.5584 0.2026 0.7834   1338.8s     0.1s
layers=100 | step

In [3]:
import numpy as np
from pyspark.ml.classification import MultilayerPerceptronClassifier, MultilayerPerceptronClassificationModel

# --- Re-entrenar el mejor modelo ---
best_mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=target_col,
    layers=[input_size, 50, 2],
    solver="gd",
    stepSize=0.1,
    maxIter=200,
    blockSize=512,
    seed=42
)

print("Re-entrenando mejor modelo (layers=50 | stepSize=0.1 | maxIter=200)...")
t0 = time.time()
best_model = best_mlp.fit(train_assembled)
print(f"Entrenamiento completado en {time.time() - t0:.1f} s")

# --- Guardar modelo en formato PySpark (para reusar con Spark) ---
MODEL_PATH = "/home/alejo/spark_project/mejor_modelo_mlp"
best_model.save(MODEL_PATH)
print(f"Modelo PySpark guardado en: {MODEL_PATH}")

# --- Extraer pesos como numpy para LIME (no necesita Spark activo) ---
# Los pesos del MLP están en best_model.weights — un vector de todos los
# pesos y biases de la red en orden: input→hidden→output
weights_vector = best_model.weights.toArray()

# Guardar con numpy — el responsable de LIME puede cargarlo sin Spark
import numpy as np
LIME_PATH = "/home/alejo/spark_project/lime_data"
import os; os.makedirs(LIME_PATH, exist_ok=True)

np.save(f"{LIME_PATH}/model_weights.npy", weights_vector)
np.save(f"{LIME_PATH}/layers.npy", np.array([input_size, 50, 2]))

# --- Guardar también una muestra del test set en numpy para que LIME tenga datos ---
# LIME necesita ejemplos reales para construir el explainer
print("Extrayendo muestra del test set para LIME...")
test_sample = (test_assembled
               .sample(fraction=0.01, seed=42)
               .select("features", target_col)
               .collect())

X_test_sample = np.array([r["features"].toArray() for r in test_sample])
y_test_sample = np.array([r[target_col] for r in test_sample])

np.save(f"{LIME_PATH}/X_test_sample.npy", X_test_sample)
np.save(f"{LIME_PATH}/y_test_sample.npy", y_test_sample)

# --- Wrapper para LIME (guardar como script .py) ---
wrapper_code = '''
import numpy as np
from pyspark.ml.linalg import Vectors

def predict_proba_wrapper(X_array, spark, model):
    """
    Wrapper para usar el modelo PySpark con LIME.
    X_array: numpy array (n_samples, n_features)
    Returns: numpy array (n_samples, 2) con [prob_clase_0, prob_clase_1]
    """
    rows = [{"features": Vectors.dense(row.tolist())} for row in X_array]
    df   = spark.createDataFrame(rows)
    preds = model.transform(df).select("probability").collect()
    return np.array([[1 - r["probability"][1], r["probability"][1]]
                     for r in preds])

# Para cargar el modelo:
# from pyspark.ml.classification import MultilayerPerceptronClassificationModel
# model = MultilayerPerceptronClassificationModel.load("/home/alejo/spark_project/mejor_modelo_mlp")

# Para cargar los datos de test:
# X_test = np.load("/home/alejo/spark_project/lime_data/X_test_sample.npy")
# y_test = np.load("/home/alejo/spark_project/lime_data/y_test_sample.npy")

# Para usar con LIME:
# import lime.lime_tabular
# explainer = lime.lime_tabular.LimeTabularExplainer(
#     training_data=X_test,
#     mode="classification",
#     class_names=["no_click", "click"]
# )
# exp = explainer.explain_instance(
#     X_test[0],
#     lambda x: predict_proba_wrapper(x, spark, model)
# )
# exp.show_in_notebook()
'''

with open(f"{LIME_PATH}/lime_wrapper.py", "w") as f:
    f.write(wrapper_code)

print(f"\nArchivos para LIME guardados en: {LIME_PATH}")
print(f"  - model_weights.npy   ({weights_vector.shape[0]} pesos)")
print(f"  - layers.npy          ([{input_size}, 50, 2])")
print(f"  - X_test_sample.npy   ({X_test_sample.shape})")
print(f"  - y_test_sample.npy   ({y_test_sample.shape})")
print(f"  - lime_wrapper.py     (instrucciones y wrapper)")
print("\nEl responsable de LIME necesita acceso a un SparkSession activo.")
print("Compartir toda la carpeta /home/alejo/spark_project/")

Re-entrenando mejor modelo (layers=50 | stepSize=0.1 | maxIter=200)...


Entrenamiento completado en 1411.2 s


Modelo PySpark guardado en: /home/alejo/spark_project/mejor_modelo_mlp
Extrayendo muestra del test set para LIME...



Archivos para LIME guardados en: /home/alejo/spark_project/lime_data
  - model_weights.npy   (4502 pesos)
  - layers.npy          ([87, 50, 2])
  - X_test_sample.npy   ((81007, 87))
  - y_test_sample.npy   ((81007,))
  - lime_wrapper.py     (instrucciones y wrapper)

El responsable de LIME necesita acceso a un SparkSession activo.
Compartir toda la carpeta /home/alejo/spark_project/
